# E024 — Audio LPC order ablation for LPCC

Vary LPC order ∈ {8, 10, 12, 14, 16, 20} for the LPCC frontend from E020.
Everything else identical: 13 cepstral coefficients, 13+Δ+ΔΔ=39d features,
UBM 32, MAP r=16, +All aug, LOSO seed=67, LLR scoring.

E020 reference (order=12): **3.33 ± 4.14% EER**, min-DCF 0.0333.

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from scipy.special import logsumexp
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
ORDERS = [8, 10, 12, 14, 16, 20]
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")
print(f"LPC orders to sweep: {ORDERS}")

## 1. Feature extraction with variable LPC order

Same `extract_lpcc` function as E020, but `order` parameterized. Cepstrum
length stays at `n_cep=13`, so the feature dimension (13+Δ+ΔΔ=39) is
identical across all orders — only the all-pole model's complexity changes.

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    lpcc_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        lpcc_frames.append(cep)
    feat   = np.array(lpcc_frames, dtype=np.float32)
    delta  = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat   = np.hstack([feat, delta, delta2])
    feat  -= feat.mean(axis=0)
    return feat


def aug_noise(y, snr_db=20.0, rng=None):
    signal_power = np.mean(y ** 2) + 1e-10
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y)).astype(y.dtype)
    return y + noise


def aug_speed(y, rate_range=(0.9, 1.1), rng=None):
    rate = rng.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)


def load_and_augment(wav_path, rng):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise(y, snr_db=20.0, rng=rng), sr),
        (aug_speed(y, rng=rng), sr),
    ]


def extract_batch(df, data_dir, seed, order):
    rng = np.random.default_rng(seed)
    all_feat, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in load_and_augment(find_wav(row["stem"], data_dir), rng):
            feat = extract_lpcc(y_aug, sr, order=order)
            all_feat.append(feat)
            all_labels.extend([row["label"]] * len(feat))
    return np.vstack(all_feat), np.array(all_labels)


def score_utterance_lpcc(wav_path, adapted, ubm, order):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = extract_lpcc(y, sr, order=order)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


def train_ubm(X, n_components=32, seed=67):
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm, X_target, r=16.0):
    log_prob   = ubm._estimate_log_prob(X_target)
    log_resp   = log_prob + np.log(ubm.weights_)
    log_resp  -= logsumexp(log_resp, axis=1, keepdims=True)
    resp       = np.exp(log_resp)
    n_k        = resp.sum(axis=0)
    mu_hat     = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha      = n_k / (n_k + r)
    adapted    = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

print("Functions defined.")

## 2. LPC order sweep — LOSO CV per order

Full 3-fold LOSO CV for each order. Train fold augmented (+All); val
utterances always scored from original WAVs. Same UBM/MAP recipe as E020.

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

results_by_order = {}  # order -> {"fold_eers": [...], "fold_dcfs": [...], "oof_scores": ndarray}

print(f"Running LPC-order ablation over {ORDERS}")
print("=" * 72)

for order in ORDERS:
    print(f"\n--- LPC order = {order} ---")
    oof_scores   = np.full(len(manifest), np.nan)
    fold_eers, fold_dcfs = [], []

    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df   = manifest.loc[val_idx]

        X_train, y_train = extract_batch(train_df, DATA, seed=SEED + fold_id, order=order)
        X_nt = X_train[y_train == 0]
        X_t  = X_train[y_train == 1]

        ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
        adapted = map_adapt(ubm, X_t, r=MAP_R)

        for idx, row in val_df.iterrows():
            oof_scores[idx] = score_utterance_lpcc(find_wav(row["stem"], DATA),
                                                   adapted, ubm, order=order)

        vs = oof_scores[val_idx]
        vl = manifest.loc[val_idx, "label"].to_numpy()
        eer, _     = compute_eer(vs[vl == 1], vs[vl == 0])
        min_dcf, _ = compute_min_dcf(vs[vl == 1], vs[vl == 0])
        fold_eers.append(eer * 100)
        fold_dcfs.append(min_dcf)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%  min-DCF={min_dcf:.4f}")

    mean_eer, std_eer = float(np.mean(fold_eers)), float(np.std(fold_eers))
    mean_dcf = float(np.mean(fold_dcfs))
    results_by_order[order] = {
        "fold_eers": fold_eers,
        "fold_dcfs": fold_dcfs,
        "mean_eer":  mean_eer,
        "std_eer":   std_eer,
        "mean_dcf":  mean_dcf,
        "oof_scores": oof_scores,
    }
    print(f"  MEAN: {mean_eer:.2f} ± {std_eer:.2f}%  min-DCF={mean_dcf:.4f}")

print("\nAll orders swept.")

## 3. Results table

In [ ]:
rows = []
for order in ORDERS:
    r = results_by_order[order]
    rows.append({
        "order":   order,
        "F0 EER": r["fold_eers"][0],
        "F1 EER": r["fold_eers"][1],
        "F2 EER": r["fold_eers"][2],
        "Mean":   r["mean_eer"],
        "Std":    r["std_eer"],
        "min-DCF": r["mean_dcf"],
    })
df_res = pd.DataFrame(rows)

best_idx = int(df_res["Mean"].idxmin())
best_order = int(df_res.loc[best_idx, "order"])

print(f"{'order':>5} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 63)
for _, row in df_res.iterrows():
    marker = " *" if int(row["order"]) == best_order else "  "
    ref    = " (E020)" if int(row["order"]) == 12 else ""
    print(f"{int(row['order']):>5}{marker} {row['F0 EER']:>7.2f} {row['F1 EER']:>7.2f} "
          f"{row['F2 EER']:>7.2f} {row['Mean']:>7.2f} {row['Std']:>7.2f} {row['min-DCF']:>9.4f}{ref}")
print("-" * 63)
print(f"* = best mean EER (order={best_order})")

best = results_by_order[best_order]
ref_e020 = results_by_order[12]
delta = best["mean_eer"] - ref_e020["mean_eer"]
print(f"\nWinner: order={best_order}  ({best['mean_eer']:.2f} ± {best['std_eer']:.2f}%, min-DCF={best['mean_dcf']:.4f})")
print(f"E020 ref order=12: {ref_e020['mean_eer']:.2f} ± {ref_e020['std_eer']:.2f}%,  min-DCF={ref_e020['mean_dcf']:.4f}")
print(f"Delta vs E020: {delta:+.2f}pp")

## 4. Mean EER ± std vs LPC order

In [ ]:
means = [results_by_order[o]["mean_eer"] for o in ORDERS]
stds  = [results_by_order[o]["std_eer"]  for o in ORDERS]
dcfs  = [results_by_order[o]["mean_dcf"] for o in ORDERS]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(ORDERS, means, yerr=stds, fmt="o-", color=COLORS["purple"],
            lw=2, capsize=5, markersize=8, label="Mean ± std EER")
# Mark winner
ax.scatter([best_order], [best["mean_eer"]], s=220, marker="*",
           color="gold", edgecolor="black", lw=1.5, zorder=10,
           label=f"optimum (order={best_order})")
# Mark E020 reference (order=12)
ax.axvline(12, color=COLORS["gray"], ls="--", lw=1.2, alpha=0.7,
           label="E020 reference (order=12)")
# Annotate values
for o, m, s in zip(ORDERS, means, stds):
    ax.text(o, m + s + 0.25, f"{m:.2f}", ha="center", fontsize=8.5,
            color=COLORS["purple"])
ax.set_xticks(ORDERS)
ax.set_xlabel("LPC order")
ax.set_ylabel("EER [%]")
ax.set_title("E024 — EER vs LPC order (LPCC 13+Δ+ΔΔ, UBM32, MAP r=16, +All)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Per-fold grouped bars across orders

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(ORDERS))
width = 0.27
fold_colors = ["#E74C3C", "#2E86AB", "#27AE60"]
for i in range(3):
    offset = (i - 1) * width
    fold_i = [results_by_order[o]["fold_eers"][i] for o in ORDERS]
    ax.bar(x + offset, fold_i, width, label=f"Fold {i}",
           color=fold_colors[i], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([str(o) for o in ORDERS])
ax.set_xlabel("LPC order")
ax.set_ylabel("EER [%]")
ax.set_title("E024 — per-fold EER across LPC orders")
ax.legend()
# Highlight winner column
winner_xi = ORDERS.index(best_order)
ax.axvspan(winner_xi - 0.5, winner_xi + 0.5, color="gold", alpha=0.12, zorder=0)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("E024 LPC-order ablation — summary")
print("=" * 60)
for o in ORDERS:
    r = results_by_order[o]
    tag = " ← winner" if o == best_order else (" (E020)" if o == 12 else "")
    print(f"order={o:>2}:  EER {r['mean_eer']:.2f} ± {r['std_eer']:.2f}%   min-DCF {r['mean_dcf']:.4f}{tag}")